In [16]:
TAILLE = 3

taquin_final = ((1,2,3),(4,5,6),(7,8,0))

taquin_initial = ((2,6,3),(7,0,5),(4,1,8))


def format_list(taquin_tuple):
    return [[ taquin_tuple[i][j] for j in range(TAILLE)] for i in range(TAILLE)]

def format_tuple(taquin_list): 
    return tuple( tuple( taquin_list[i][j] for j in range(TAILLE)) for i in range(TAILLE))


def haut(taquin,i0,j0):
    """ renvoie un nouveau taquin au format tuple dans lequel le 0 a bougé vers le haut """
    taq_list = format_list(taquin)
    if i0>0:
        taq_list[i0][j0]   =  taq_list[i0-1][j0]
        taq_list[i0-1][j0] = 0
    return format_tuple(taq_list)

def bas(taquin,i0,j0):
    """ renvoie un nouveau taquin au format tuple dans lequel le 0 a bougé vers le bas"""
    taq_list = format_list(taquin)
    if i0<TAILLE - 1:
        taq_list[i0][j0]   =  taq_list[i0+1][j0]
        taq_list[i0+1][j0] = 0
    return format_tuple(taq_list)
    

def gauche(taquin,i0,j0):
    """ renvoie un nouveau taquin au format tuple dans lequel le 0 a bougé vers la gauche"""
    taq_list = format_list(taquin)
    if j0>0:
        taq_list[i0][j0]   =  taq_list[i0][j0-1]
        taq_list[i0][j0-1] = 0
    return format_tuple(taq_list)

def droite(taquin,i0,j0):
    """ renvoie un nouveau taquin au format tuple dans lequel le 0 a bougé vers la droite"""
    taq_list = format_list(taquin)
    if j0<TAILLE - 1:
        taq_list[i0][j0]   =  taq_list[i0][j0+1]
        taq_list[i0][j0+1] = 0
    return format_tuple(taq_list)

def voisins(taquin) : 
    """cherche la position du 0 dans le taquin
    et renvoie la liste des configurations voisines"""
    for i in range(len(taquin)):
        for j in range(len(taquin[i])):
            if taquin[i][j] == 0:
                i0, j0 = i, j
    resu = []
    if i0>0:
        resu.append(haut(taquin,i0,j0))
    if i0 < TAILLE-1:
        resu.append(bas(taquin,i0,j0))
    if j0>0:
        resu.append(gauche(taquin,i0,j0))
    if j0<TAILLE - 1:
        resu.append(droite(taquin,i0,j0))
    return resu

def parcours_largeur(depart):
    '''parcours en largeur depuis un taquin de depart
    la fonction renvoie un dictionnaire avec
    - comme clés : les configurations accessibles depuis le taquin depart
    - comme valeurs : la configuration immédiatement précédente'''
    dist = {depart:None}
    courant = [depart]  # liste des sommets à une distance 'n'
    suivant = []        # liste des sommets à une distance 'n+1'
    while taquin_final not in dist:
        s = courant.pop()  #on retire un sommet à la distance n
        for v in voisins(s): 
            if v not in dist:
                dist[v] = s
                suivant.append(v)
        if len(courant)==0:
            suivant.reverse()  # juste pour l'esthétique... on reverse la pile
            courant = suivant
            suivant = []
    return dist

def construire_solution(depart):
    """la fonction renvoie la liste des configurations depuis depart jusqu'au taquin final"""
    d = parcours_largeur(depart)
    sol = []
    t = taquin_final
    while t != None : 
        sol.append(t)
        t = d[t]
    sol.reverse()
    return sol

def afficher_solution(depart):
    i = 0
    for s in construire_solution(depart):
        for ligne in s : 
            print(ligne)
        i+=1
        print(' - - - - - ', i)

# Ajout d'une métrique
la fonction renvoie un minorant du nombre de déplacements élémentaires permettant de passer de la configuration depart à arrivee

In [17]:
def metrique(taquin_depart, taquin_arrivee):
    dico_depart = {}
    dico_arrivee = {}
    for i in range(TAILLE):
        for j in range(TAILLE):
            dico_depart[taquin_depart[i][j]] = (i,j)
            dico_arrivee[taquin_arrivee[i][j]] = (i,j)
    resu = 0
    for k in dico_depart :
        if k != 0:
            resu += abs(dico_depart[k][0]-dico_arrivee[k][0]) 
            resu += abs(dico_depart[k][1]-dico_arrivee[k][1])
    return resu
    

In [5]:
metrique(taquin_initial, taquin_final)

({2: (0, 0), 6: (0, 1), 3: (0, 2), 7: (1, 0), 0: (1, 1), 5: (1, 2), 4: (2, 0), 1: (2, 1), 8: (2, 2)}, {1: (0, 0), 2: (0, 1), 3: (0, 2), 4: (1, 0), 5: (1, 1), 6: (1, 2), 7: (2, 0), 8: (2, 1), 0: (2, 2)})

In [10]:
metrique(taquin_final, taquin_final)

0

In [14]:
metrique(taquin_initial, bas(taquin_initial,1,1))

1

In [23]:
def parcours_largeur_metrique(depart, arrivee, OK = True):
    '''parcours en largeur depuis un taquin de depart
    la fonction renvoie un dictionnaire avec
    - comme clés : les configurations accessibles depuis le taquin depart
    - comme valeurs : la configuration immédiatement précédente'''
    dist = {depart:None}
    courant = [depart]  # liste des sommets à une distance 'n'
    suivant = {}        # DICTIONNAIRE de listes de sommets à distance 'n+1', rangés par métrique jusqu'à l'arrivée.
    while arrivee not in dist and OK:
        s = courant.pop()  #on retire un sommet à la distance n
        for v in voisins(s): 
            if v not in dist:
                dist[v] = s
                metr = metrique(v,arrivee)
                # ajout d'une nouvelle valeur dans le dictionnaire
                if metr in suivant : 
                    suivant[metr].append(v)
                else : 
                    suivant[metr] = [v]
        if len(courant)==0:
            courant = []
            for metr in suivant :
                for v in suivant[metr] : 
                    courant.append(v)
            suivant = {}
    return dist

def afficher_solution(depart):
    i = 0
    d = parcours_largeur_metrique(taquin_final, depart)
    t = depart
    while t != None : 
        for ligne in t : 
            print(ligne)
        print('---')
        t = d[t]

In [20]:
d = parcours_largeur_metrique(taquin_initial, taquin_final)

In [24]:
afficher_solution(taquin_initial)

(2, 6, 3)
(7, 0, 5)
(4, 1, 8)
---
(2, 6, 3)
(7, 1, 5)
(4, 0, 8)
---
(2, 6, 3)
(7, 1, 5)
(0, 4, 8)
---
(2, 6, 3)
(0, 1, 5)
(7, 4, 8)
---
(2, 6, 3)
(1, 0, 5)
(7, 4, 8)
---
(2, 0, 3)
(1, 6, 5)
(7, 4, 8)
---
(2, 3, 0)
(1, 6, 5)
(7, 4, 8)
---
(2, 3, 5)
(1, 6, 0)
(7, 4, 8)
---
(2, 3, 5)
(1, 0, 6)
(7, 4, 8)
---
(2, 3, 5)
(1, 4, 6)
(7, 0, 8)
---
(2, 3, 5)
(1, 4, 6)
(7, 8, 0)
---
(2, 3, 5)
(1, 4, 0)
(7, 8, 6)
---
(2, 3, 0)
(1, 4, 5)
(7, 8, 6)
---
(2, 0, 3)
(1, 4, 5)
(7, 8, 6)
---
(0, 2, 3)
(1, 4, 5)
(7, 8, 6)
---
(1, 2, 3)
(0, 4, 5)
(7, 8, 6)
---
(1, 2, 3)
(4, 0, 5)
(7, 8, 6)
---
(1, 2, 3)
(4, 5, 0)
(7, 8, 6)
---
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)
---
